# 01 - Rigorous Data Validation & Quality Scoring

## 1. Introduction & Reproducibility
To guarantee research validity, we evaluate the dataset across 5 dimensions: Schema, Referential Integrity, Temporal Logic, Business Rules, and Statistical Outliers. We calculate a precise **Data Quality Score** and log these experiments using `MLflow` for full reproducibility.



In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import mlflow
import time
from datetime import datetime
from scipy.stats import skew, kurtosis

# Reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

# Load Augmented Data
tx_df = pd.read_parquet('../data/raw/transactions.parquet')
acct_df = pd.read_parquet('../data/raw/accounts.parquet')

dq_metrics = {}
validation_evidence = []
start_time = time.time()

# Initialize MLflow run
mlflow.set_experiment("AegisAML_Data_Quality")
run = mlflow.start_run(run_name="Data_Validation_v1")
print(f"MLflow Run ID: {run.info.run_id}")



## 2. Schema Validation (Completeness & Consistency)
We assess missing values and strictly enforce datatypes.



In [ ]:
required_cols = ['tx_id', 'sender_id', 'receiver_id', 'amount', 'timestamp', 'is_sar', 'typology']
missing_cols = set(required_cols) - set(tx_df.columns)
passed_schema = len(missing_cols) == 0

null_counts = tx_df.isnull().sum()
total_cells = tx_df.shape[0] * tx_df.shape[1]
completeness = 1.0 - (null_counts.sum() / total_cells)
dq_metrics['Completeness'] = completeness * 100

# Type Consistency (amount must be float, IDs must be numeric or string, timestamp must be datetime)
try:
    tx_df['timestamp'] = pd.to_datetime(tx_df['timestamp'])
    tx_df['amount'] = tx_df['amount'].astype(float)
    consistency = 100.0
except Exception as e:
    consistency = 0.0

dq_metrics['Consistency'] = consistency

validation_evidence.append({'Check': 'Schema Check', 'Passed': passed_schema, 'Detail': 'All columns present'})
validation_evidence.append({'Check': 'Null Values', 'Passed': null_counts.sum() == 0, 'Detail': f"{null_counts.sum()} total nulls"})

display(pd.DataFrame(validation_evidence))



## 3. Referential Integrity
Graph algorithms require absolute integrity. Every sender and receiver must exist in the accounts table.



In [ ]:
invalid_senders = ~tx_df['sender_id'].isin(acct_df['account_id'])
invalid_receivers = ~tx_df['receiver_id'].isin(acct_df['account_id'])

total_tx = len(tx_df)
failed_integrity = invalid_senders.sum() + invalid_receivers.sum()
integrity_score = (total_tx - failed_integrity) / total_tx
dq_metrics['Integrity'] = integrity_score * 100

validation_evidence.append({'Check': 'Ref Integrity', 'Passed': failed_integrity == 0, 'Detail': f"{failed_integrity} orphaned edges"})
print(f"Integrity Score: {dq_metrics['Integrity']:.2f}%")



## 4. Temporal Validation
Checking for timestamps in the future or logically impossible orderings.



In [ ]:
future_txs = tx_df[tx_df['timestamp'] > pd.Timestamp.now()].shape[0]
timeliness_score = (total_tx - future_txs) / total_tx
dq_metrics['Timeliness'] = timeliness_score * 100

validation_evidence.append({'Check': 'Temporal (Future)', 'Passed': future_txs == 0, 'Detail': f"{future_txs} future txs"})



## 5. Business Rule Validation
Validating logic constraints:
1. Transfer amount must be positive.
2. Sender cannot equal Receiver (Self-transfer).



In [ ]:
neg_amt = (tx_df['amount'] <= 0).sum()
self_tx = (tx_df['sender_id'] == tx_df['receiver_id']).sum()

failed_validity = neg_amt + self_tx
validity_score = (total_tx - failed_validity) / total_tx
dq_metrics['Validity'] = validity_score * 100

validation_evidence.append({'Check': 'Positive Amounts', 'Passed': neg_amt == 0, 'Detail': f"{neg_amt} negative/zero txs"})
validation_evidence.append({'Check': 'No Self-Transfers', 'Passed': self_tx == 0, 'Detail': f"{self_tx} self-transfers"})

evidence_df = pd.DataFrame(validation_evidence)
display(evidence_df)



## 6. Statistical & Distribution Validation
We leverage exploratory statistical techniques (ECDF, IQR Outliers, Box Plots) to understand the distribution profile of normal vs. SAR transactions.



In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle("Statistical Exploratory Analysis", fontsize=18)

# 1. Log-scale Distribution
sns.histplot(tx_df, x='amount', hue='is_sar', bins=50, log_scale=True, ax=axes[0, 0])
axes[0, 0].set_title("Log-Scale Transaction Amount Distribution")

# 2. Box Plot for Outliers
sns.boxplot(data=tx_df, x='is_sar', y='amount', ax=axes[0, 1])
axes[0, 1].set_yscale('log')
axes[0, 1].set_title("Box Plot (Log Scale) by SAR Label")

# 3. ECDF Plot
sns.ecdfplot(data=tx_df, x='amount', hue='is_sar', ax=axes[1, 0])
axes[1, 0].set_xscale('log')
axes[1, 0].set_title("ECDF of Transaction Amounts")

# 4. Typology Distribution
typology_counts = tx_df['typology'].value_counts()
axes[1, 1].pie(typology_counts, labels=typology_counts.index, autopct='%1.1f%%', startangle=90, colors=sns.color_palette('pastel'))
axes[1, 1].set_title("Typology Distribution")

plt.tight_layout()
plt.savefig('../figures/statistical_validation.png')
plt.show()

# IQR Outlier Analysis
Q1 = tx_df['amount'].quantile(0.25)
Q3 = tx_df['amount'].quantile(0.75)
IQR = Q3 - Q1
outliers = tx_df[(tx_df['amount'] < (Q1 - 1.5 * IQR)) | (tx_df['amount'] > (Q3 + 1.5 * IQR))]
print(f"IQR Outliers Detected: {len(outliers)} ({(len(outliers)/total_tx)*100:.2f}%)")



## 7. Data Quality Score & MLflow Logging
We calculate the final rigorous Quality Score and commit the metrics to our MLflow experiment tracker.



In [ ]:
final_dq_score = np.mean(list(dq_metrics.values()))

print("=========================================")
print(" RIGOROUS DATA QUALITY SCORE REPORT      ")
print("=========================================")
for k, v in dq_metrics.items():
    print(f"{k:15}: {v:.2f}%")
print("-----------------------------------------")
print(f"OVERALL QUALITY: {final_dq_score:.2f}%")
print("=========================================")

# Log to MLflow
mlflow.log_metric("Completeness", dq_metrics['Completeness'])
mlflow.log_metric("Consistency", dq_metrics['Consistency'])
mlflow.log_metric("Integrity", dq_metrics['Integrity'])
mlflow.log_metric("Timeliness", dq_metrics['Timeliness'])
mlflow.log_metric("Validity", dq_metrics['Validity'])
mlflow.log_metric("Overall_Quality_Score", final_dq_score)

mlflow.log_param("Total_Transactions", total_tx)
mlflow.log_param("Total_Accounts", len(acct_df))
mlflow.log_param("Validation_Failures", evidence_df[~evidence_df['Passed']].shape[0])



## 8. Final Export & Metadata Footer
Export validated data and close the tracking run.



In [ ]:
# Only keep strictly valid transactions
clean_tx_df = tx_df[(tx_df['amount'] > 0) & (tx_df['sender_id'] != tx_df['receiver_id']) & (tx_df['sender_id'].isin(acct_df['account_id'])) & (tx_df['receiver_id'].isin(acct_df['account_id']))]

clean_tx_df.to_parquet('../data/processed/transactions_clean.parquet', index=False)
print(f"Exported {len(clean_tx_df)} clean transactions.")

end_time = time.time()
mlflow.log_param("Execution_Time_Seconds", round(end_time - start_time, 2))
mlflow.end_run()

print("--- Notebook Metadata ---")
print(f"Dataset Version: v1.0_clean")
print(f"Execution Time: {end_time - start_time:.2f} seconds")
print("MLflow Run completed successfully.")

